# N11 Tape Seed Ablation

Runs seed ablations for the tape model family using the `2d_tape_ICNN.ipynb` configuration. The default focuses on the anisotropic structured Brazier ICNN model; uncomment the alternate architecture list to broaden the sweep.

In [1]:
import os
from pathlib import Path

import jax
import jax.numpy as jnp

jax.config.update("jax_enable_x64", True)
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")

from properties import TapeN11Properties
from run_architectures import SweepConfig

ROOT = Path.cwd()
train_file = "../experiment_data/tape_data/11_noded/n11_tape_train_dataset.npz"
valid_file = "../experiment_data/tape_data/11_noded/n11_tape_test_dataset.npz"
properties = TapeN11Properties(mass=-0.005)

# Copied from 2d_tape_ICNN.ipynb
K_init_chol = (0.2, 0.0, 0.1)
K_init_diag = (0.2, 0.1)

base_cfg = SweepConfig(
    der_K_diag=K_init_diag,
    der_K_chol=K_init_chol,
    hidden=(10,),
    corr_factor=0.01,
    input_mode="raw",
    only_stretching_NN=False,
    only_bending_NN=False,
    zero_reference=True,
    activation="tanh",
    mode="anisotropic",
    n_epochs=1000,
    lr=5e-2,
    weight_decay=1e-5,
    seed=42,
    valid_every=10,
    max_dlambda=1e-2,
    iters=10,
    ls_steps=10,
    abs_tol=5e-4,
    rel_tol=1e-4,
    early_stop=True,
    train_fail_on_nonconvergence=False,
    prediction_fail_on_nonconvergence=False,
    hessian_reg_strength=1e-6,
    hessian_reg_probes=1,
    hessian_reg_seed=0,
    force_key=None,
    force_loss_strength=0.0,
    force_components=(0, 1, 2),
    force_sign=1.0,
    return_loss_components=False,
    early_stopping=True,
    early_stopping_patience=200,
    early_stopping_min_delta=1e-5,
    restore_best_model=True,
    save_npz=True,
    save_model=True,
    save_plots=True,
    save_force_predictions=False,
    plot_force_predictions=False,
    save_hessian_diagnostics=False,
    save_energy_landscapes=False,
    verbose=True,
    continue_on_failure=True,
    seed_list=tuple(range(26)),
)

print(properties)

OUTPUT_DIR = ROOT / "seed_ablation_outputs_n11_tape"
SUMMARY_DIR = OUTPUT_DIR / "seed_ablation_summary"
base_cfg = base_cfg.__class__(**{
    **base_cfg.__dict__,
    "output_dir": str(OUTPUT_DIR),
    "seed_list": base_cfg.seed_list,
})
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Saving seed ablation results under: {OUTPUT_DIR.resolve()}")


TapeN11Properties(length=None, r0=0.005, axs=None, jxs=None, ixs1=None, ixs2=None, density=600.0, E=1000000.0, N=11, start=Array([0., 0., 0.], dtype=float64), end=Array([1.05660479, 0.        , 0.04239192], dtype=float64), mass=-0.005)
Saving seed ablation results under: /Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/seed_ablation_outputs_n11_tape


In [ ]:
from run_architectures import subset_brazier_stiffness_only, subset_tape_tube_candidates

# Broader tape seed ablation options:
selected_architectures = [
    "diag_energy_mlp",
    "chol_energy_mlp",
    "diag_energy_icnn",
    "chol_energy_icnn",
    "diag_stiffness_mlp",
    "chol_stiffness_mlp",
    "brazier_chol_stiffness_mlp",
    "brazier_chol_stiffness_icnn",
]
# selected_architectures = subset_tape_tube_candidates()

print(f"Running {len(selected_architectures)} architectures across seeds {base_cfg.seed_list}:")
for name in selected_architectures:
    print(f"  - {name}")


Running 8 architectures across seeds (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25):
  - diag_energy_mlp
  - chol_energy_mlp
  - diag_energy_icnn
  - chol_energy_icnn
  - diag_stiffness_mlp
  - chol_stiffness_mlp
  - brazier_chol_stiffness_mlp
  - brazier_chol_stiffness_icnn


: 

In [ ]:
from seed_ablation_utils import run_seed_ablation

all_seed_results = run_seed_ablation(
    properties=properties,
    train_file=train_file,
    valid_file=valid_file,
    base_cfg=base_cfg,
    selected_architectures=selected_architectures,
)



SEED ABLATION for architecture: diag_energy_mlp

--- Running seed 0 for diag_energy_mlp ---

Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 0
  n_epochs                : 1000
  lr                      : 0.05
  weight_decay            : 1e-05
  max_dlambda             : 0.01
  iters                   : 10
  ls_steps                : 10
  abs_tol                 : 0.0005
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  restore_best_mo

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/util.py:664: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


Epoch 000 | Train total: 2.305e-02 | Train disp: 1.911e-02 | Train force: 0.000e+00 | Valid total: 2.806e-02 | Valid disp: 2.806e-02 | Valid force: 0.000e+00
Epoch 100 | Train total: 5.340e-03 | Train disp: 4.558e-03 | Train force: 0.000e+00 | Valid total: 1.041e-02 | Valid disp: 1.041e-02 | Valid force: 0.000e+00
Epoch 200 | Train total: 4.647e-03 | Train disp: 4.098e-03 | Train force: 0.000e+00 | Valid total: 2.389e-02 | Valid disp: 2.389e-02 | Valid force: 0.000e+00
Early stopping at epoch 210 | Best epoch: 010 | Best valid: 5.512e-03


/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/run_architectures.py:1038: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)
/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/architecture_plots.py:234: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()



--- Running seed 1 for diag_energy_mlp ---

Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 1
  n_epochs                : 1000
  lr                      : 0.05
  weight_decay            : 1e-05
  max_dlambda             : 0.01
  iters                   : 10
  ls_steps                : 10
  abs_tol                 : 0.0005
  rel_tol                 : 0.0001
  early_stop              : True
  training fail_on_nonconvergence       : False
  validation loss fail_on_nonconvergence: False
  prediction fail_on_nonconvergence     : False
  early_stopping          : True
  early_stopping_patience : 200
  restore_best_model      : True
  hessian_reg_strength    : 1e-06

/Users/radha/GitRepos/dismech-jax/examples/slinky/slinky_2D/util.py:664: RuntimeWarning: Mass is non-positive; if this is because gravity is in +z, ignore, but if this is not the intention please correct.
  base, aux = get_slinky(properties)


Epoch 000 | Train total: 2.305e-02 | Train disp: 1.911e-02 | Train force: 0.000e+00 | Valid total: 2.778e-02 | Valid disp: 2.778e-02 | Valid force: 0.000e+00
Epoch 100 | Train total: 5.230e-03 | Train disp: 4.465e-03 | Train force: 0.000e+00 | Valid total: 1.004e-02 | Valid disp: 1.004e-02 | Valid force: 0.000e+00
Epoch 200 | Train total: 4.489e-03 | Train disp: 3.934e-03 | Train force: 0.000e+00 | Valid total: 1.162e-02 | Valid disp: 1.162e-02 | Valid force: 0.000e+00
Early stopping at epoch 210 | Best epoch: 010 | Best valid: 5.653e-03

--- Running seed 2 for diag_energy_mlp ---

Running architecture: diag_energy_mlp
  model_cls               : DiagonalPlusEnergyNN
  which_case              : MLP
  hidden                  : (10,)
  input_mode              : raw
  only_stretching_NN      : False
  only_bending_NN         : False
  activation              : tanh
  corr_factor             : 0.01
  zero_reference          : True
  seed                    : 2
  n_epochs                : 1

In [ ]:
import importlib
import seed_ablation_utils as sau

importlib.reload(sau)

# Load saved per-seed results; this does not retrain.
all_seed_results = sau.load_seed_ablation_results(OUTPUT_DIR)

sau.print_seed_summary(all_seed_results)
sau.save_seed_summary_json(all_seed_results, output_dir=str(SUMMARY_DIR))

sau.make_seed_ablation_plots(
    all_seed_results=all_seed_results,
    output_dir=str(SUMMARY_DIR),
    traj_idx=0,
)

In [ ]:
from seed_ablation_utils import run_seed_hessian_diagnostics

run_seed_hessian_diagnostics(
    str(OUTPUT_DIR),
    use_predicted=True,
    splits=("train", "valid"),
    max_trajectories=1,
    stride=10,
    properties_class="TapeN11Properties",
)
print("Hessian diagnostics table:", OUTPUT_DIR / "hessian_diagnostics_table.csv")


In [ ]:
print("Done. Key outputs:")
print("  - <architecture>__seed_*/results.npz")
print("  - seed_ablation_summary/*.json and *.png")
print("  - hessian_diagnostics_table.csv")
